***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.5 小角近似的失效与 $w$ 项](5_5_widefield_effect.ipynb)
    * 下一节：[5.x 延伸阅读与参考文献](5_x_further_reading_and_references.ipynb)

***


## 5.6 成像参数选择的物理依据

第 5 章前几节已经分别讨论空间频率、采样函数、网格化、权重和宽场效应。真实成像时，这些概念会同时出现为一组成像参数：`cell`、`imsize`、`weighting`、`robust`、`uvtaper`、`gridder`、`deconvolver`、`scales`、`nterms`、`threshold` 和 mask。若把它们看成互不相关的菜单，成像很容易退化为试错；若把它们看成同一个反问题的不同约束，就能从科学目标反推参数选择。

成像参数的本质是决定三个边界。第一，图像要表达哪些角尺度和视场范围；第二，数据中哪些空间频率和频率结构被赋予更高权重；第三，算法允许天空模型具有哪些自由度。参数选择不只是得到一张好看的图，而是定义最终科学量的测量条件。

![成像参数决策图](figures/imaging_parameter_decision_map.png)

**图 5.6.1** 成像参数是一组耦合决策。科学目标首先决定角尺度、视场、灵敏度和动态范围要求，随后才进入像素尺度、图像大小、权重、taper、gridder、deconvolver、mask 和 QA 产品。

### 5.6.1 像素尺度与图像大小

像素尺度 `cell` 的基本约束来自合成波束采样。若合成波束主轴量级为 $	heta_{m beam}$，常用选择是让一个波束宽度覆盖约 3 到 5 个像素：

$$
\Delta	heta_{m pix}\lesssim {	heta_{m beam}\over 3	ext{--}5}.
$$

像素过大时，峰值位置、峰值通量和源形态会被离散采样误差污染；像素过小时，图像矩阵变大，噪声像素数量增加，去卷积自由度也随之增多。过小的像素并不会创造新的角分辨率，因为角分辨率仍由 `uv` 覆盖和权重决定。

图像大小 `imsize` 的约束来自视场，而视场不只由目标本体大小决定。主波束内的亮源、旁瓣污染源、mosaic 重叠区、宽场 `w` 项误差和 primary beam correction 后的噪声放大都可能要求更大的成像范围。若图像边界切掉亮源旁瓣，去卷积会把边界之外的真实信号留在残差中，随后污染目标区域。

![像素尺度与波束采样](figures/cellsize_beam_sampling.png)

**图 5.6.2** 像素尺度应充分采样合成波束。过粗像素会损失峰值和位置精度；适度采样能稳定表示波束形状；过度采样增加计算量和模型自由度，却不提高真实角分辨率。

### 5.6.2 权重、taper 与表面亮度灵敏度

可见度权重决定脏波束和图像噪声。自然权重通常给高采样密度区域更大贡献，因此热噪声较低，但分辨率较粗、旁瓣结构受阵列采样密度影响明显。均匀权重压低密集 `uv` 区域的贡献，可提高分辨率并改变旁瓣，却会牺牲点源灵敏度。Briggs robust 权重在两者之间连续调节。`uvtaper` 则显式压低长基线权重，用角分辨率换取扩展结构和低面亮度成分的灵敏度。

这些选择没有普适最优。若科学目标是紧致核位置或双源分离，需要强调长基线和较小波束；若目标是弥散晕、星系盘总通量或分子云大尺度结构，过度追求高分辨率反而会降低表面亮度信噪比。成像时常需要输出至少两类图像：一类保持较高分辨率用于结构定位，另一类使用 taper 或自然权重用于扩展通量测量。

![权重与 taper 的取舍](figures/weighting_taper_tradeoff_summary.png)

**图 5.6.3** 权重和 taper 同时改变噪声、波束大小和旁瓣。左图用相对量示意自然、Briggs、均匀和 taper 的典型取舍；右图显示不同权重策略会改变 PSF 主瓣宽度和旁瓣结构。

### 5.6.3 Gridder 与宽场宽带算法边界

当视场较小、带宽较窄、主波束变化可以忽略时，普通二维 gridder 加小角近似已经足够。视场增大后，$w$ 项相位误差随视场半径平方增长，需要 W-projection、W-stacking 或 faceting。主波束随方向和频率显著变化时，A-projection 或 AW-projection 才能把方向相关响应纳入成像算子。宽带连续谱成像还需要处理频谱结构，MT-MFS 用 Taylor 展开把天空亮度写成频率的低阶函数；若源谱结构复杂、主波束模型不足或不同频率的 `uv` 覆盖差异过大，谱指数图会带有明显系统偏差。

算法选择应从误差项是否进入科学量判断。若只需窄场点源峰值通量，复杂宽场算法可能没有必要；若目标位于主波束边缘、视场内有强离轴源，或科学量依赖谱指数和低面亮度结构，忽略这些项会把方向相关误差写入图像。计算成本也是科学设计的一部分：更复杂的 gridder 和更多 Taylor 项会提高计算量，也会增加模型自由度和诊断需求。

![成像算法边界](figures/imaging_algorithm_boundary_map.png)

**图 5.6.4** 成像算法选择随视场和频谱复杂性改变。窄场窄带问题可用普通 gridder 和简单 deconvolver；宽场、宽带、mosaic 和方向相关效应会逐步要求 W/AW-projection、faceting、MT-MFS、多尺度模型和更严格的验证。

### 5.6.4 阈值、mask 与停止准则

去卷积阈值和 mask 决定哪些残差信号被允许进入模型。阈值过高会残留真实源旁瓣，阈值过低会把噪声和校准残差吸收到天空模型。mask 过紧会截断扩展结构，mask 过松会给噪声峰和旁瓣提供自由度。自动 mask 可以提高效率，但其行为仍受噪声估计、旁瓣结构和多尺度模型影响，不能替代残图检查。

比较稳健的做法，是把停止准则连接到图像质量产品。残图 RMS 是否接近理论热噪声，残差直方图是否接近对称，强源周围是否仍有相干旁瓣，主波束边缘是否噪声放大，源通量是否随阈值继续系统变化，这些问题比单个 `niter` 数值更重要。对于低面亮度目标，最好比较不同 mask、taper 和尺度列表下的通量恢复，而不是只报告最深一次 CLEAN 的结果。

### 5.6.5 教学案例：同一数据的两张正确图像

一组连续谱数据同时包含一个紧致核和一片低面亮度延展发射。若只做均匀权重高分辨率图像，紧致核位置和形态很清楚，但延展发射被分解成低信噪比斑块；若只做自然权重加 taper 图像，延展结构变得显著，紧致核却被较大的波束混合。两张图像并不是互相矛盾，而是测量了同一数据中不同空间频率权重下的科学量。

这个案例应输出一组配套产物：高分辨率图用于紧致结构定位，taper 图用于扩展通量，残图用于判断是否过度去卷积，PSF 图用于说明旁瓣和分辨率，参数表记录 cell、imsize、weighting、robust、taper、gridder、deconvolver、threshold 和 mask。最终报告应说明哪个科学量来自哪张图，不能把高分辨率图的峰值信噪比和 taper 图的积分通量混在同一个误差预算中。

### 5.6.6 本节结论

成像参数不是软件习惯，而是科学问题、Fourier 采样、仪器响应和反问题先验之间的接口。`cell` 与 `imsize` 决定图像能否正确表示波束和视场；权重与 taper 决定分辨率、噪声和表面亮度灵敏度；gridder 与 deconvolver 决定哪些几何、宽带和方向相关效应被纳入模型；threshold 与 mask 决定模型自由度何时停止增长。成像结果的可信度来自这些选择与科学目标的一致性，以及残图、PSF、噪声和通量稳定性的共同验证。

***

下一节：[5.x 延伸阅读与参考文献](5_x_further_reading_and_references.ipynb)
